# 06.4 — The EMA prior normalization deep dive

[06.1](06.1_multi_task_losses_overview.ipynb) showed the problem: the four loss components live on wildly different scales, so a raw sum is dominated by whichever is largest. This notebook is the fix — **EMA prior normalization** (Critical Notes #6, #30). Each component is divided by its own *running average magnitude* (bringing it to ~1.0), then rescaled to a shared reference (classification), then weighted. The result: the *weights* control the balance, not the accidental scales.

This notebook covers:

* The normalization idea: divide by a running magnitude.
* The EMA update — a running average that adapts as scales drift over training.
* Why classification is the reference (Critical Note #30).
* `LossPriors`, `aggregate_normalized_losses`, and first-iteration bootstrapping.

**Prerequisites:** [06.1 overview](06.1_multi_task_losses_overview.ipynb), [04.7 weighted loss](../04_architecture/04.7_weighted_classification_loss.ipynb).

## Section 1 — What MATLAB does

`cgg_getLossInformation` + `cgg_processLossComponent` maintain a running "prior" magnitude per component and normalize against it:

```matlab
% Update the running magnitude (EMA) for each component:
Prior_Rec = PriorProportion * Prior_Rec + (1-PriorProportion) * mean(Loss_Rec);

% Normalize each loss by its prior, rescale to classification's, weight:
Rescale = Prior_Classification;                          % the shared reference
Loss_Rec_norm = Weight_Rec * (Loss_Rec / Prior_Rec) * Rescale;
```

`PriorProportion = 0.9` sets how fast the running average adapts. The reference is classification's prior — every component is scaled to *its* magnitude. The port reproduces this exactly, and the empirical parity probe corrected the plan's original (wrong) note about the normalization — [06.10](06.10_nan_masked_reconstruction.ipynb) tells that story.

## Section 2 — The Python concepts you need

### 2.1 — Normalize by a running magnitude

The core idea: divide each loss by an estimate of its own typical size, so every component becomes ~1.0 regardless of its natural scale. Then a shared multiplier (the reference) and the weight take over. Watch four wildly-scaled losses collapse to a common scale:

In [ ]:
import torch
from neural_data_decoding.training.losses.multi_objective import aggregate_normalized_losses, LossPriors

# The 06.1 §2.2 problem: components differing by orders of magnitude.
recon = torch.tensor(1200.0, requires_grad=True)
kl    = torch.tensor(3.5,    requires_grad=True)
cls   = torch.tensor(1.1,    requires_grad=True)

priors = LossPriors.initial()          # no history yet — first iteration bootstraps
out = aggregate_normalized_losses(
    reconstruction_loss=recon, kl_loss=kl, classification_loss=cls,
    weights={"reconstruction": 100, "kl": 1, "classification": 10},
    priors=priors, prior_proportion=0.9,
)
print(f"reference (Rescale_Value = classification's prior): {out.rescale_value.item():.2f}")
print(f"normalized reconstruction: {out.reconstruction.item():.1f}   (was 1200!)")
print(f"normalized KL            : {out.kl.item():.2f}")
print(f"normalized classification: {out.classification.item():.1f}")
print(f"total: {out.total.item():.1f}")

The magic: the raw reconstruction was 1200; normalized it's 110 — comparable to classification's 11. Each component was divided by its own magnitude (→ ~1.0), multiplied by the shared reference (1.1 = classification's scale), then by its weight (100, 1, 10). Now the **ratio 100:1:10** in the *weights* is what determines balance — exactly as intended, not the accidental 1200:3.5:1.1 of the raw losses.

The arithmetic, spelled out:
- reconstruction: `100 × (1200/1200) × 1.1 = 110`
- KL: `1 × (3.5/3.5) × 1.1 = 1.1`
- classification: `10 × (1.1/1.1) × 1.1 = 11`

Each `(loss / its_prior)` is ≈1.0 on the first iteration (the prior bootstraps to the current loss), so the weight × reference is what remains.

### 2.2 — The EMA update: adapting to drift

In [ ]:
# Priors are RUNNING averages — they adapt as loss scales change over training.
# EMA: new_prior = proportion·old + (1-proportion)·current
proportion = 0.9
prior = 100.0                          # current running magnitude
observations = [100, 100, 50, 50, 50]  # loss halves partway through training
for obs in observations:
    prior = proportion * prior + (1 - proportion) * obs
    print(f"  observed {obs:3} → prior now {prior:.2f}")
print("→ the prior drifts smoothly toward the new scale, not jumping to each batch's value.")

The EMA (exponential moving average) means the prior tracks the *trend* of each component's magnitude, ignoring per-batch noise. `prior_proportion = 0.9` keeps 90% of the old estimate each step, so it adapts over ~10 iterations rather than lurching. This matters because loss scales genuinely change during training (reconstruction shrinks as the decoder improves) — a fixed normalization would drift out of calibration; the EMA follows.

### 2.3 — Why classification is the reference

The normalization scales everything to *classification's* magnitude (Critical Note #30). Why classification and not, say, reconstruction? Because classification is the task that matters — the deliverable is decoding accuracy. Anchoring the common scale to classification means the weights express "how much does each auxiliary loss matter *relative to the task*." Reconstruction and KL are means to a better classifier; scaling them to the classifier's magnitude keeps them in service of it.

When classification is absent (a pure autoencoder Stage 1, [05.6](../05_training_loop/05.6_the_two_stage_lifecycle.ipynb)), the reference falls back to reconstruction — the notebook's `_determine_rescale_value` handles that.

### 2.4 — First-iteration bootstrapping

In [ ]:
# On iteration 1 there's no history, so each prior bootstraps to the current loss.
# That makes every (loss / prior) exactly 1.0 on the first batch:
fresh = LossPriors.initial()
print("before first batch — classification prior:", fresh.classification)   # None

out = aggregate_normalized_losses(
    reconstruction_loss=torch.tensor(1200.0), classification_loss=torch.tensor(1.1),
    weights={"reconstruction": 100, "classification": 10},
    priors=fresh, prior_proportion=0.9,
)
print("after first batch — updated classification prior:", out.updated_priors.classification.item())
print("→ the prior seeds itself from batch 1, then EMAs from there.")

The `updated_priors` come back from the aggregator for the training loop to carry into the next iteration — the running state threads through the epoch, exactly like the confidence history ([06.6](06.6_confidence_routing.ipynb)) and the checkpoint bookkeeping ([05.5](../05_training_loop/05.5_checkpoint_resume_state_machine.ipynb)). The first-iteration self-seeding matches MATLAB's behavior and avoids a divide-by-None on batch one.

## Section 3 — The `neural_data_decoding` implementation

`_process_component` — the normalize → rescale → weight pipeline for one component:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/losses/multi_objective.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("def _process_component"))
body = next(j for j in range(i, len(src)) if "raw_detached" in src[j])
for k in range(body, min(body + 16, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* The prior is updated via EMA (`prior_proportion` blend) and **detached** — it's a constant scaling factor in the gradient, not something to backprop through ([02.5 §2.4](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb)).
* The processed loss preserves gradient flow w.r.t. its *input* (the actual loss) while the normalization factors are detached constants — so the gradient sees `weight × rescale / prior × loss`, and only `loss` carries grad.
* `_determine_rescale_value` (above this function) picks classification's prior as the reference, falling back to reconstruction — §2.3's logic.
* The docstrings cite `cgg_processLossComponent` lines — parity is traced line-to-line, and the empirical probe (not the plan's text) is the source of truth for the exact normalization ([06.10](06.10_nan_masked_reconstruction.ipynb)).

## Section 4 — Hands-on exercises

### Exercise 4.1 — normalization makes weights meaningful

Run the aggregator with weights 100:1:10, then with 1:1:100, on the SAME raw losses. Show the total's composition shifts to follow the WEIGHTS, not the raw scales.

In [ ]:
# Reveal:
for w in [{"reconstruction": 100, "kl": 1, "classification": 10},
          {"reconstruction": 1, "kl": 1, "classification": 100}]:
    out = aggregate_normalized_losses(
        reconstruction_loss=torch.tensor(1200.0), kl_loss=torch.tensor(3.5),
        classification_loss=torch.tensor(1.1), weights=w, priors=LossPriors.initial())
    print(f"weights {list(w.values())}: recon={out.reconstruction.item():.1f}, "
          f"class={out.classification.item():.1f} → the weight ratio now governs the balance")

### Exercise 4.2 — the EMA smooths noise

Feed a noisy sequence of magnitudes through the §2.2 EMA and show the prior is far smoother than the raw observations (lower variance).

In [ ]:
# Reveal:
import statistics
torch.manual_seed(0)
obs = [100 + 40*torch.randn(1).item() for _ in range(30)]     # noisy around 100
prior, priors_seq = 100.0, []
for o in obs:
    prior = 0.9 * prior + 0.1 * o
    priors_seq.append(prior)
print(f"raw observations std:  {statistics.pstdev(obs):.1f}")
print(f"EMA prior std:         {statistics.pstdev(priors_seq):.1f}  (much smoother)")

## Section 5 — Common errors

### One component still dominates after "normalizing"

The priors aren't being carried across iterations — each call gets a fresh `LossPriors.initial()`, so the EMA never accumulates. Thread `out.updated_priors` back in each step (§2.4).

### The rescale reference is wrong

If classification is present it should anchor the scale (§2.3). A run that normalizes to reconstruction's scale when classification is active has the fallback logic inverted — check `_determine_rescale_value`.

### Gradient explodes when a prior is tiny

Dividing by a near-zero prior blows up the normalized loss. Early training, or a component that's genuinely near zero, can trigger this — the EMA smoothing (§2.2) mitigates, and clipping ([05.3](../05_training_loop/05.3_gradient_clipping.ipynb)) is the safety net.

### Weights that worked in MATLAB behave differently

The weights act on the *post-normalization* scale (§2.1). If normalization isn't active (or uses a different reference/proportion), the same weights produce a different balance. Confirm `prior_proportion=0.9` and that priors thread through.

### Backprop through the prior

The prior is a *constant* scaling factor and must be detached ([02.5 §2.4](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb)). Backpropping through it would let the model game the normalization by shrinking the prior instead of the loss.

## Section 6 — Further reading

- [06.12 ema prior normalization deep dive](06.12_ema_prior_normalization_deep_dive.ipynb) — the cross-component mechanics and first-iteration degeneracy in more depth.
- [Exponential moving average](https://en.wikipedia.org/wiki/Moving_average#Exponential_moving_average) — the smoothing this uses.
- [`src/neural_data_decoding/training/losses/multi_objective.py`](../../src/neural_data_decoding/training/losses/multi_objective.py) — `LossPriors`, `_process_component`, `aggregate_normalized_losses`.

Next up: **[06.5 MIL softmax pooling](06.5_mil_softmax_pooling.ipynb)** — Multiple Instance Learning, and why the classification softmax pools over multiple axes.